# HW4.a + HW4.5 on one GPU (Colab / Kaggle / lab PC)

Follows the HW4.5 clarifications: **one GPU is enough**, **Part C is optional** (skipped here; METRICS.md then says
"Part C skipped (optional)"), and HW4.5 uses **the same data as HW4.a**: Twitter US Airline Sentiment, rows from
SLICE = 605, 70/15/15 split with SEED = 1605. The HW4.a run writes `data/hw4a_data_lock.json`, and every later run
refuses to train if its rows or split differ.

Colab: *Runtime → Change runtime type → GPU*. Kaggle: *Accelerator → GPU T4 (x1 is enough)*, *Internet on*.
Upload this project folder as a zip (`hw45.zip`) before running the cells. Every cell appends to `RUN_LOG.txt`.

In [ ]:
# Clarification 3: record which GPU was used
!nvidia-smi -L
!nvidia-smi

In [ ]:
# Unpack the project (hw45.zip uploaded to /content, /kaggle/working, or attached as Kaggle input)
import glob, os, zipfile
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content' if os.path.exists('/content') else os.getcwd()
found = glob.glob(f'{WORK}/**/run_card.sh', recursive=True)
attached = glob.glob('/kaggle/input/**/run_card.sh', recursive=True)  # Kaggle unzips uploaded datasets (read-only)
if not found and attached:
    import shutil
    shutil.copytree(os.path.dirname(attached[0]), f'{WORK}/hw45')
    found = [f'{WORK}/hw45/run_card.sh']
if not found:
    zips = glob.glob(f'{WORK}/*.zip') + glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if not zips and os.path.exists('/content'):
        from google.colab import files
        files.upload()
        zips = glob.glob('/content/*.zip')
    zipfile.ZipFile(zips[0]).extractall(f'{WORK}/hw45')
    found = glob.glob(f'{WORK}/hw45/**/run_card.sh', recursive=True)
%cd {os.path.dirname(found[0])}
!chmod +x *.sh

In [ ]:
# Versions that work on a T4 (verified on Kaggle 2026-11-03): vLLM 0.11.0 + torch 2.8.0+cu128 + transformers 4.57.
# vLLM 0.11 needs transformers < 5 (its tokenizer code uses all_special_tokens_extended, removed in 5.x).
# The pip "dependency conflicts" warning lists Kaggle/Colab extras this assignment never imports; it is safe to ignore.
# A restarted session loses pip installs: rerun this cell first every time.
!pip install -q "vllm==0.11.0" "transformers<5" "bitsandbytes>=0.45" kagglehub scikit-learn pandas matplotlib
# The preinstalled torchaudio is built for another CUDA and breaks `import transformers`; audio is not used here
!pip uninstall -y -q torchaudio
!python -c "import torch, transformers, vllm; from transformers import BertForSequenceClassification; print('torch', torch.__version__, '| transformers', transformers.__version__, '| vllm', vllm.__version__, '| cuda ok:', torch.cuda.is_available())"

In [ ]:
# Same Twitter dataset as HW4.a (prints the file's sha256)
!python get_data.py

In [ ]:
# Card label for results/<card>/ , e.g. colab_t4, kaggle_t4, local_rtx4090
import re, subprocess
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader', '-i', '0'], text=True).strip()
platform = 'kaggle' if os.path.exists('/kaggle') else 'colab' if os.path.exists('/content') else 'local'
CARD = platform + '_' + re.sub(r'nvidia|geforce|tesla|[^a-z0-9]', '', gpu.lower())
print(gpu, '->', CARD)

HW4.a baseline (also writes the data lock). Then HW4.a vLLM inference and serving (Table HW4.1).

In [ ]:
!./run_card.sh {CARD} env hw4a

In [ ]:
# The T4 (compute 7.5) has no FlashAttention 2 and vLLM's FlexAttention fallback crashes on it; use Triton attention
%env VLLM_ATTENTION_BACKEND=TRITON_ATTN
!./run_card.sh {CARD} serve

## Part A: batch scaling to the memory wall (each batch size in its own process)

In [ ]:
!./run_card.sh {CARD} partA

## Part B: AMP, gradient checkpointing, gradient accumulation, 8-bit AdamW, then the best combination

In [ ]:
!./run_card.sh {CARD} partB

## Part D: serving and KV cache on this card

(The KV test uses Qwen2.5-7B on a 24 GB+ card and Qwen2.5-1.5B on a 16 GB T4; override with `KV_MODEL=...`.)

In [ ]:
!./run_card.sh {CARD} partD

## Part C (optional, skipped)

Only if you have two GPUs at once, e.g. Kaggle *GPU T4 x2*. Uncomment to run it; otherwise METRICS.md says "Part C skipped (optional)".

In [ ]:
# !./run_ddp.sh kaggle_t4x2

## Report: figures/ and METRICS.md, then a zip to download (models/ excluded)

In [ ]:
!python make_report.py
!zip -qr hw45_results.zip METRICS.md RUN_LOG.txt results env figures data/hw4a_data_lock.json
print('download', os.path.abspath('hw45_results.zip'))